# 🔍 Day 6 — Evaluation + Calibration
**Vision System Capstone v3** | ECE · Temperature Scaling · Grad-CAM · MC Dropout

This notebook covers:
1. Load best checkpoint correctly (with hooks)
2. ECE before calibration
3. Temperature scaling (fit on val, eval on test)
4. ECE after calibration — before/after comparison
5. Grad-CAM grid — correct + misclassified
6. MC Dropout uncertainty analysis
7. Day 6 summary

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Load configs ──────────────────────────────────────────────────────────────
with open('../configs/data_config.yaml')  as f: DCFG = yaml.safe_load(f)
with open('../configs/model_config.yaml') as f: MCFG = yaml.safe_load(f)

NUM_CLASSES  = DCFG['dataset']['num_classes']
CLASSES_FILE = Path('../') / DCFG['dataset']['class_names_file']
CLASS_NAMES  = [c.strip() for c in CLASSES_FILE.read_text().splitlines() if c.strip()]

print(f'Dataset    : {DCFG["dataset"]["name"]}')
print(f'Classes    : {NUM_CLASSES}')
print(f'Class file : {CLASSES_FILE}')

---
## 1. Load Best Checkpoint (with Grad-CAM hooks)

In [ ]:
# ── Always use load_checkpoint_with_hooks — never model.load_state_dict() ─────
from models.resnet import build_model
from utils.checkpoint import load_checkpoint_with_hooks

CKPT_PATH = '../logs/checkpoints/best.pth'

model = build_model(
    model_cfg_path='../configs/model_config.yaml',
    data_cfg_path ='../configs/data_config.yaml',
).to(device)

if Path(CKPT_PATH).exists():
    ckpt = load_checkpoint_with_hooks(CKPT_PATH, model, device=device)
    print(f'Checkpoint loaded | epoch={ckpt["epoch"]} | best_acc={ckpt["best_acc"]:.4f}')
    print(f'Grad-CAM hook active: {model._gradcam_hook is not None} ✓')
else:
    print('⚠ best.pth not found — using untrained model')
    print('  Run training first: python training/train.py')

model.eval()
print(f'Model params: {model.count_params():,}')

In [ ]:
# ── Build DataLoaders ─────────────────────────────────────────────────────────
from datasets.dataset_loader import get_dataloaders, load_config as load_data_cfg

data_cfg    = load_data_cfg('../configs/data_config.yaml')
loaders     = get_dataloaders(data_cfg)
val_loader  = loaders['val']
test_loader = loaders['test']

print(f'Val  batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

---
## 2. ECE Before Calibration

In [ ]:
# ── Collect predictions on test set ──────────────────────────────────────────
from evaluation.calibration import collect_predictions, compute_ece, reliability_diagram

print('Collecting predictions on test set...')
confs_raw, preds_raw, labels = collect_predictions(model, test_loader, device)

ece_before = compute_ece(confs_raw, preds_raw, labels)
acc_before = (preds_raw == labels).mean()

print(f'\nBefore calibration:')
print(f'  Accuracy : {acc_before:.4f}')
print(f'  ECE      : {ece_before:.4f}')
print(f'  Mean conf: {confs_raw.mean():.4f}')
if ece_before > 0.1:
    print(f'  ⚠ ECE > 0.10 — model is overconfident. Temperature scaling needed.')
else:
    print(f'  ✓ ECE acceptable')

---
## 3. Temperature Scaling

In [ ]:
# ── Fit T on val set ──────────────────────────────────────────────────────────
from evaluation.calibration import TemperatureScaling

print('Fitting Temperature Scaling on val set...')
temp_scaler = TemperatureScaling()
learned_T   = temp_scaler.fit(model, val_loader, device)

print(f'\nLearned temperature T = {learned_T:.4f}')
if learned_T > 1:
    print(f'  T > 1 → model was overconfident — probabilities softened ✓')
elif learned_T < 1:
    print(f'  T < 1 → model was underconfident — probabilities sharpened')
else:
    print(f'  T ≈ 1 → model already well calibrated')

---
## 4. ECE After Calibration — Before/After Comparison

In [ ]:
# ── ECE after calibration ─────────────────────────────────────────────────────
confs_cal, preds_cal, _ = collect_predictions(
    model, test_loader, device, temp_scaler
)

ece_after = compute_ece(confs_cal, preds_cal, labels)
acc_after = (preds_cal == labels).mean()

print(f'After calibration:')
print(f'  Accuracy : {acc_after:.4f}')
print(f'  ECE      : {ece_after:.4f}')
print(f'\nImprovement:')
print(f'  ECE: {ece_before:.4f} → {ece_after:.4f} '
      f'({100*(ece_before-ece_after)/ece_before:.1f}% reduction)')

# Save scaler
import torch
Path('../logs').mkdir(exist_ok=True)
torch.save({'temperature': learned_T}, '../logs/temperature_scaler.pt')
print('\nTemperature scaler saved → logs/temperature_scaler.pt')

In [ ]:
# ── Reliability diagrams side by side ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

reliability_diagram(
    confs_raw, preds_raw, labels,
    title='Before Temperature Scaling', ax=axes[0]
)
reliability_diagram(
    confs_cal, preds_cal, labels,
    title='After Temperature Scaling', ax=axes[1]
)

fig.suptitle(
    f'Calibration: ECE {ece_before:.4f} → {ece_after:.4f}  |  T = {learned_T:.4f}',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
Path('../logs').mkdir(exist_ok=True)
plt.savefig('../logs/calibration_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → logs/calibration_comparison.png')

---
## 5. Grad-CAM Grid

In [ ]:
# ── Generate Grad-CAM grid (10 correct + 10 wrong) ───────────────────────────
from evaluation.gradcam import visualize_grid

test_dataset = loaders['test'].dataset

print('Generating Grad-CAM grid...')
print('(This may take 1-2 mins on CPU)')

visualize_grid(
    model       = model,
    dataset     = test_dataset,
    class_names = CLASS_NAMES,
    cfg         = DCFG,
    device      = device,
    n_correct   = 10,
    n_wrong     = 10,
    save_path   = '../logs/gradcam_outputs/gradcam_grid.png',
)

# Display saved image
from PIL import Image as PILImage
grid_path = Path('../logs/gradcam_outputs/gradcam_grid.png')
if grid_path.exists():
    img = PILImage.open(grid_path)
    plt.figure(figsize=(16, 20))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Grad-CAM Grid', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 6. MC Dropout Uncertainty

In [ ]:
# ── Evaluate uncertainty on test set ──────────────────────────────────────────
from evaluation.uncertainty import evaluate_uncertainty, plot_uncertainty_distribution

print('Running MC Dropout (N=20 passes, 500 samples)...')
print('(~2-3 mins on CPU)')

unc_results = evaluate_uncertainty(
    model       = model,
    loader      = test_loader,
    device      = device,
    n_passes    = 20,
    max_samples = 500,
)

print(f'\nUncertainty results:')
print(f'  Flagged (std > 0.15) : {unc_results["flagged_frac"]*100:.1f}% of samples')
print(f'  Accuracy (all)       : {unc_results["accuracy_all"]:.4f}')
print(f'  Accuracy (certain)   : {unc_results["accuracy_certain"]:.4f}')
print(f'\nInterpretation:')
print(f'  Certain samples acc > All samples acc = MC Dropout correctly')
print(f'  identifies hard samples worth flagging for human review.')

In [ ]:
# ── Uncertainty distribution plot ─────────────────────────────────────────────
plot_uncertainty_distribution(
    unc_results,
    save_path='../logs/uncertainty_distribution.png'
)

from PIL import Image as PILImage
unc_path = Path('../logs/uncertainty_distribution.png')
if unc_path.exists():
    img = PILImage.open(unc_path)
    plt.figure(figsize=(16, 5))
    plt.imshow(img); plt.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# ── Single image MC Dropout demo ──────────────────────────────────────────────
from evaluation.uncertainty import mc_dropout_predict, plot_single_uncertainty

# Take one test image
img_tensor, true_label = test_dataset[0]

result = mc_dropout_predict(model, img_tensor, device, n_passes=20)

print(f'MC Dropout single image result:')
print(f'  True class  : {CLASS_NAMES[true_label]}')
print(f'  Predicted   : {CLASS_NAMES[result["pred_class"]]}')
print(f'  Confidence  : {result["confidence"]:.4f}')
print(f'  Uncertainty : {result["uncertainty"]:.4f}')
print(f'  Flag review : {result["is_uncertain"]}')

plot_single_uncertainty(
    result=result,
    class_names=CLASS_NAMES,
    save_path='../logs/single_uncertainty_demo.png'
)

from PIL import Image as PILImage
demo_path = Path('../logs/single_uncertainty_demo.png')
if demo_path.exists():
    img = PILImage.open(demo_path)
    plt.figure(figsize=(8, 5))
    plt.imshow(img); plt.axis('off')
    plt.tight_layout(); plt.show()

---
## 7. Day 6 Summary

In [ ]:
# ── Full summary table ────────────────────────────────────────────────────────
print('=' * 60)
print('DAY 6 SUMMARY')
print('=' * 60)
print(f'  Accuracy         : {acc_before:.4f}')
print()
print('  CALIBRATION:')
print(f'    ECE before      : {ece_before:.4f}')
print(f'    ECE after       : {ece_after:.4f}')
print(f'    Temperature T   : {learned_T:.4f}')
print(f'    Improvement     : {100*(ece_before-ece_after)/ece_before:.1f}%')
print()
print('  UNCERTAINTY:')
print(f'    Flagged samples : {unc_results["flagged_frac"]*100:.1f}%')
print(f'    Accuracy certain: {unc_results["accuracy_certain"]:.4f}')
print(f'    Accuracy all    : {unc_results["accuracy_all"]:.4f}')
print()
print('  PLOTS SAVED:')
for f in [
    'logs/calibration_comparison.png',
    'logs/gradcam_outputs/gradcam_grid.png',
    'logs/uncertainty_distribution.png',
    'logs/temperature_scaler.pt',
]:
    exists = '✓' if Path(f'../{f}').exists() else '✗'
    print(f'    {exists} {f}')
print()
print('INTERVIEW ANSWERS:')
print('  Calibration:')
print(f'    "ECE dropped from {ece_before:.2f} to {ece_after:.2f} after temperature')
print(f'     scaling with T={learned_T:.2f}. The model was overconfident — pushing')
print('     confidence close to 1.0 even on hard samples."')
print()
print('  Grad-CAM:')
print('    "Grad-CAM revealed the model was focusing on plate edges instead')
print('     of food texture. Fixed with CutMix — forces attention to full plate."')
print()
print('  MC Dropout:')
print(f'    "MC Dropout flags {unc_results["flagged_frac"]*100:.0f}% of predictions as uncertain.')
print('     Accuracy on certain samples is higher — these flagged predictions')
print('     feed our data flywheel for weekly retraining."')